In [68]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-6 ; m=1; M=10 ; g=9.8 ; R=2
tf=15; N=int(tf/dt);
theta0 = np.pi/4              #condiciones iniciales

In [59]:
#Funciones

def Cart(R,theta):
  xM= theta*R
  yM= R
  xm = xM - R*np.sin(theta)
  ym = yM - R*np.cos(theta)
  return xm, ym, xM, yM

def Ec_EuLag(M,m,R,g,theta,wm):
    dwdt=-(1/(( M + 2*m*(1-np.cos(theta)))*R)) * ( m*R*wm*wm*np.sin(theta) + m*g*np.sin(theta))
    return dwdt
  
def Motion(M,m,R,g,theta,wm,am,t):
    for i in range(len(t)-1):
        dt = t[i+1] - t[i]
        # predictores de aceleración, velocidad y posición (Euler)
        am_pred = Ec_EuLag(M,m,R,g,theta[i],wm[i])  # aceleración

        wm_pred = wm[i] + am_pred * dt  # Euler vx(i+1)=vx(i)+a_pred*dt ;
        theta_pred = theta[i] + wm[i] * dt  # Euler x(i+1)=x(i)+vx_pred*dt ;

        # correctores de posición, aceleración y velocidad (HEUN)
        theta[i+1] = theta[i] + 0.5 * (wm[i] + wm_pred) * dt
        am_next = Ec_EuLag(M,m,R,g,theta[i+1],wm[i+1])
        wm[i+1] = wm[i] + 0.5 * (am_pred + am_next) * dt

    return theta, wm, am

#Derivada Total
def Dx(x,f):
#   dydx = np.zeros(np.size(f))
#   dydx[0]=(f[1]-f[0])/(x[1]-x[0])
#   dydx[-1]=(f[-1]-f[-2])/(x[-1]-x[-2])
#   dydx[1:-1]=(f[2:]-f[:-2])/(x[2:]-x[:-2])
    dydx = np.gradient(f, x)
    return dydx

#Derivada Parcial
def Dparcial(f, theta, wm, var='wm', h=1e-3):
    """Derivada parcial de f(theta, wm) por diferencias centrales.
    theta y wm son np.ndarray del mismo tamaño; f opera sobre arrays (np.cos, ...).
    var='wm'    -> df/d(theta_punto), con theta fijo
    var='theta' -> df/d(theta),       con wm fijo"""
    if var == 'wm':
        return (f(theta, wm + h) - f(theta, wm - h)) / (2*h)
    elif var == 'theta':
        return (f(theta + h, wm) - f(theta - h, wm)) / (2*h)
    raise ValueError("var debe ser 'wm' o 'theta'")

In [20]:
#Inicialización
t=np.linspace(0,tf,N)

rm , rM = np.zeros((2, N)) , np.zeros((2, N))
theta, wm, am = (np.zeros(N) for _ in range(3))
theta[0]=theta0

am[0] = Ec_EuLag(M,m,R,g,theta[0],wm[0])


In [21]:
#Principal loop
theta, wm, am =Motion(M,m,R,g,theta,wm,am,t)
rm[0,:] , rm[1,:], rM[0,:], rM[1,:] = Cart(R,theta)

In [71]:
#Energia y Hamilltoniano

def Lagrangiano(theta, wm):
    T = m*(R**2)*(wm**2)*(1 - np.cos(theta)) + 0.5*M*(R**2)*(wm**2)
    V = m*g*R*(1 - np.cos(theta)) + M*g*R
    return T - V

T=m*(R**2)*(wm**2)*(1-np.cos(theta)) + 0.5*M*(R**2)*(wm**2)
V=m*g*R*(1-np.cos(theta)) + M*g*R
E = T + V

L = Lagrangiano(theta, wm)                         #hay que definir el lagrangiano como funcion                   
p = Dx_parcial(Lagrangiano, theta, wm, var='wm')   #hay que hacerlo con la derivada parcial, no total, porque L depende de theta y wm 
H = p*wm - L

In [69]:
#Ejemplo con sympy
th, w = sp.symbols('theta omega')
m_, M_, R_, g_ = sp.symbols('m M R g', positive=True)

L_sym = m_*R_**2*w**2*(1 - sp.cos(th)) + sp.Rational(1, 2)*M_*R_**2*w**2 \
        - m_*g_*R_*(1 - sp.cos(th)) - M_*g_*R_

p_sym = sp.diff(L_sym, w)                    # dL/d(theta_punto), parcial de verdad
H_sym = p_sym*w - L_sym

p_num = sp.lambdify((th, w, m_, M_, R_, g_), p_sym, 'numpy')
H_num = sp.lambdify((th, w, m_, M_, R_, g_), H_sym, 'numpy')

p = p_num(theta, wm, m, M, R, g)             # entran y salen arrays
H = H_num(theta, wm, m, M, R, g)

In [ ]:
#Gráfica: Energía vs Hamiltoniano en función del tiempo
s = max(1, N // 20000)            # submuestreo para no dibujar millones de puntos

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(t[::s], E[::s], '-',  color='tab:blue', lw=2, label='Energía')
ax1.plot(t[::s], H[::s], '--', color='tab:red',  lw=1.5, label='Hamiltoniano')
ax1.set(ylabel='Energía (J)', title='Energía y Hamiltoniano en función del tiempo')
ax1.ticklabel_format(useOffset=False)
ax1.set_ylim(0.9995*E[0], 1.0001*E[0])
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(t[::s], E[::s] - H[::s], color='tab:green', lw=1)
ax2.set(xlabel='t (s)', ylabel='E − H (J)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#Animación
T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 30       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 3.0      # duración de la estela de la masa m (s); 0 = sin estela

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

xm, ym = rm[0, idx], rm[1, idx]                      # masa m (borde de la rueda)
xM, yM = rM[0, idx], rM[1, idx]                      # masa M (centro de la rueda)

# Figura
fig, ax = plt.subplots(figsize=(8, 5))
ax.set(xlim=(xM.min() - 1.3*R, xM.max() + 1.3*R), ylim=(-0.3*R, 2.3*R), aspect='equal',
       xlabel='x (m)', ylabel='y (m)', title='Masa en el borde de una rueda que rueda sin deslizar')
ax.grid(alpha=0.3)
ax.axhline(0, color='k', lw=2)                        # suelo

phi = np.linspace(0, 2*np.pi, 200)
wheel,  = ax.plot([], [], 'k-', lw=1)                 # rueda de radio R
spoke,  = ax.plot([], [], 'k--', lw=1)                # radio: centro -> masa m
center, = ax.plot([], [], 'o', color='tab:blue', ms=9)   # masa M
bob,    = ax.plot([], [], 'o', color='tab:red', ms=9)    # masa m
trail,  = ax.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)
time_txt = ax.text(0.03, 0.92, '', transform=ax.transAxes)

def update(k):
    wheel.set_data(xM[k] + R*np.cos(phi), yM[k] + R*np.sin(phi))
    spoke.set_data([xM[k], xm[k]], [yM[k], ym[k]])
    center.set_data([xM[k]], [yM[k]])
    bob.set_data([xm[k]], [ym[k]])
    if trail_n > 0:
        j = max(0, k - trail_n)
        trail.set_data(xm[j:k+1], ym[j:k+1])
    time_txt.set_text(f't = {t[idx[k]]:.2f} s')
    return wheel, spoke, center, bob, trail, time_txt

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())